# Server trên Wi-Fi

Máy server: 172.20.10.12. Chạy hai ô code đầu rồi chờ client. Không Run All vì ô cuối đóng kết nối. Không cần chạy nhanh giữa các ô.


## 1 Tạo socket lắng nghe



In [ ]:
import socket
import threading
HOST = '0.0.0.0'
PORT = 5000
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind((HOST, PORT))
server.listen()
server.settimeout(0.5)
clients = {}
clients_lock = threading.Lock()
workers = []
stop = threading.Event()
print('Server:', server.getsockname())


## 2 Nhận client và chuyển tiếp tin



In [ ]:
def handle_client(conn, address):
    try:
        with conn.makefile('rb') as reader:
            while True:
                raw = reader.readline(4097)
                if not raw:
                    break
                if len(raw) > 4096 or not raw.endswith(b'\n'):
                    break
                text = raw[:-1].decode('utf-8')
                message = f'[{address[1]}] {text}\n'.encode('utf-8')
                print(message.decode('utf-8').rstrip())
                with clients_lock:
                    targets = list(clients.items())
                for target, write_lock in targets:
                    try:
                        with write_lock:
                            target.sendall(message)
                    except OSError:
                        pass
    except (OSError, UnicodeError):
        pass
    finally:
        with clients_lock:
            clients.pop(conn, None)
        conn.close()
def accept_clients():
    while not stop.is_set():
        try:
            conn, address = server.accept()
        except socket.timeout:
            continue
        except OSError:
            break
        with clients_lock:
            clients[conn] = threading.Lock()
        worker = threading.Thread(target=handle_client,
                                  args=(conn, address), daemon=True)
        workers.append(worker)
        worker.start()
accept_thread = threading.Thread(target=accept_clients, daemon=True)
accept_thread.start()
print('Đang chờ client. Bây giờ mở notebook Client A và Client B.')


## 3 Dừng server

Chỉ chạy khi kết thúc buổi thực hành.


In [ ]:
stop.set()
server.close()
accept_thread.join(2)
with clients_lock:
    remaining = list(clients)
for conn in remaining:
    try:
        conn.shutdown(socket.SHUT_RDWR)
    except OSError:
        pass
    conn.close()
for worker in workers:
    worker.join(2)
print('Đã dừng server')
